# Inference Optimization Challenge — Qwen2.5-0.5B on Tesla T4

**Goal:** Maximize output token throughput (tok/s) for Qwen2.5-0.5B on a Tesla T4 GPU.  
**Baseline to beat:** 3,332 tok/s

## Scoring

| Metric | Weight | Constraint |
|--------|--------|------------|
| Output throughput (tok/s) | 40% | Higher is better |
| P99 TPOT | 20% | Must stay under 50 ms |
| P99 TTFT | 15% | Must stay under 2000 ms |
| Request success rate | 10% | Must be 100% |
| Code quality & documentation | 15% | Clean, reproducible, explained |

## 1. Setup

In [1]:
!pip install -q vllm==0.15.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB

In [ ]:
!nvidia-smi

## 2. Launch vLLM Server (Baseline — Default Config)

In [2]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-0.5B"

# Launch vLLM server in the background
server_proc = subprocess.Popen(
    ["vllm", "serve", MODEL],
    stdout=open("server_baseline.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Server PID: {server_proc.pid}")

# Wait for server to be ready
for i in range(120):
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"Server ready after {i*2}s")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print("ERROR: Server not ready after 240s")
    print(open("server_baseline.log").read()[-2000:])

Server PID: 5061
ERROR: Server not ready after 240s
.
(EngineCore_DP0 pid=5356) 
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
(EngineCore_DP0 pid=5356) 
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.12s/it]
(EngineCore_DP0 pid=5356) 
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.12s/it]
(EngineCore_DP0 pid=5356) 
(EngineCore_DP0 pid=5356) INFO 04-07 13:12:54 [default_loader.py:291] Loading weights took 1.15 seconds
(EngineCore_DP0 pid=5356) INFO 04-07 13:12:55 [gpu_model_runner.py:4130] Model loading took 0.93 GiB memory and 41.417003 seconds
(EngineCore_DP0 pid=5356) INFO 04-07 13:13:03 [backends.py:812] Using cache directory: /root/.cache/vllm/torch_compile_cache/399056c7bb/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=5356) INFO 04-07 13:13:03 [backends.py:872] Dynamo bytecode transform time: 6.85 s
(EngineCore_DP0 pid=5356) INFO 04-07 13:13:13 [backends.py:302] Cache the g

## 3. Run Benchmark (Baseline)

In [5]:
!pip install h2loop_bench

In [6]:
from h2loop_bench import result

result(name="Test", email="you@example.com",
contact_number="+91XXXXXXXXXX",
colab_link="https://colab.research.google.com/...",
port=8000 # port your vLLM server is running on
)

Running benchmark against http://localhost:8000/v1 ...
BASELINE RESULTS
  Output throughput:  3319.84 tok/s
  Mean TPOT:          13.88 ms
  P99 TPOT:           14.91 ms  (limit: 50 ms)
  Mean TTFT:          601.76 ms
  P99 TTFT:           1357.42 ms  (limit: 2000 ms)
  Completed requests: 200/200
  Failed requests:    0

Submitting results to H2Loop...
Submission successful! Record ID: recvsIfItXhXVMxds


## 5. Cleanup

In [ ]:
server_proc.terminate()
server_proc.wait()
print("Server stopped.")